In [1]:
from keplergl import KeplerGl
import webbrowser
import os
import json
import datetime
import pandas as pd

In [2]:
date = "2026_03_04"
with open(f"../vehicle_positions_data/{date}/processed.json", "r") as f:
    raw_data = json.load(f)

In [3]:
flat_data = []

# 3. Loop through the nested hierarchy
# Step into the top-level keys (e.g., "1")
for key, route_info in raw_data.items():
    route_id = route_info.get("routeId")
    
    # Step into the list of trips
    for trip in route_info.get("trips", []):
        trip_id = trip.get("tripId")
        
        # vehicleIds is a list, so we grab the first item (if it exists)
        vehicle_ids = trip.get("vehicleIds", [])
        vehicle_id = vehicle_ids[0] if vehicle_ids else "Unknown"
        
        # Step into the actual GPS coordinates
        for pos in trip.get("vehiclePositions", []):
            
            # Build a flat dictionary for this exact moment in time
            flat_data.append({
                "route_id": route_id,
                "trip_id": trip_id,
                "vehicle_id": vehicle_id,
                "latitude": pos.get("latitude"),
                "longitude": pos.get("longitude"),
                "speed": pos.get("speed"),
                "bearing": pos.get("bearing"),
                "raw_timestamp": pos.get("timestamp")
            })

# 4. Convert the list of flat dictionaries into a Pandas DataFrame
df = pd.DataFrame(flat_data)
df

,route_id,trip_id,vehicle_id,latitude,longitude,speed,bearing,raw_timestamp
0,1,2959611_0001,2864,30.244345,-97.729973,0.000000,0.000000,1772619444
1,1,2959611_0001,2864,30.244347,-97.729965,0.000000,0.000000,1772619459
2,1,2959611_0001,2864,30.244347,-97.729965,0.000000,0.000000,1772619474
3,1,2959611_0001,2864,30.244347,-97.729973,0.000000,0.000000,1772619492
4,1,2959611_0001,2864,30.244347,-97.729973,0.000000,0.000000,1772619504
...,...,...,...,...,...,...,...,...
1096252,990,2975033_22557,9403,30.331980,-97.722481,15.199360,29.000000,1772671959
1096253,990,2975033_22557,9403,30.334545,-97.720879,15.288768,28.900000,1772671980
1096254,990,2975033_22557,9403,30.335520,-97.720261,13.768831,28.400000,1772671989
1096255,990,2975033_22557,9403,30.336861,-97.719406,9.119616,28.799999,1772672004


In [4]:
# Convert the raw seconds into a Kepler-friendly formatted string
df['timestamp'] = (
    pd.to_datetime(df['raw_timestamp'], unit='s')
    .dt.tz_localize('UTC')
    .dt.tz_convert('America/Chicago')
    .dt.strftime('%Y-%m-%d %H:%M:%S')
)

df

,route_id,trip_id,vehicle_id,latitude,longitude,speed,bearing,raw_timestamp,timestamp
0,1,2959611_0001,2864,30.244345,-97.729973,0.000000,0.000000,1772619444,2026-03-04 04:17:24
1,1,2959611_0001,2864,30.244347,-97.729965,0.000000,0.000000,1772619459,2026-03-04 04:17:39
2,1,2959611_0001,2864,30.244347,-97.729965,0.000000,0.000000,1772619474,2026-03-04 04:17:54
3,1,2959611_0001,2864,30.244347,-97.729973,0.000000,0.000000,1772619492,2026-03-04 04:18:12
4,1,2959611_0001,2864,30.244347,-97.729973,0.000000,0.000000,1772619504,2026-03-04 04:18:24
...,...,...,...,...,...,...,...,...,...
1096252,990,2975033_22557,9403,30.331980,-97.722481,15.199360,29.000000,1772671959,2026-03-04 18:52:39
1096253,990,2975033_22557,9403,30.334545,-97.720879,15.288768,28.900000,1772671980,2026-03-04 18:53:00
1096254,990,2975033_22557,9403,30.335520,-97.720261,13.768831,28.400000,1772671989,2026-03-04 18:53:09
1096255,990,2975033_22557,9403,30.336861,-97.719406,9.119616,28.799999,1772672004,2026-03-04 18:53:24


In [5]:
df = df.sort_values(by='timestamp', ascending=True)

# 2. Reset the index so the row numbers count up cleanly from 0 again
df = df.reset_index(drop=True)
df

,route_id,trip_id,vehicle_id,latitude,longitude,speed,bearing,raw_timestamp,timestamp
0,335,2965692_9490,2552,30.288250,-97.702850,12.070080,119.000000,1772603978,2026-03-03 23:59:38
1,325,2965206_8978,2716,30.400379,-97.703049,12.517119,30.000000,1772603978,2026-03-03 23:59:38
2,486,2967834_13268,2858,30.255842,-97.745499,0.000000,0.000000,1772603978,2026-03-03 23:59:38
3,801,2973071_20408,5015,30.386669,-97.683983,20.563839,19.000000,1772603978,2026-03-03 23:59:38
4,837,2974003_22385,2614,30.312407,-97.665543,13.768831,207.199997,1772603978,2026-03-03 23:59:38
...,...,...,...,...,...,...,...,...,...
1096252,4,2975989_12072,2405,30.257824,-97.704514,0.000000,0.000000,1772690380,2026-03-04 23:59:40
1096253,7,2972283_19820,2627,30.213388,-97.745697,6.884416,301.299988,1772690380,2026-03-04 23:59:40
1096254,486,2967834_13268,2858,30.258310,-97.748093,14.547832,318.000000,1772690380,2026-03-04 23:59:40
1096255,217,2961776_3909,2361,30.236290,-97.700996,7.599680,118.000000,1772690380,2026-03-04 23:59:40


In [6]:
# KEPLER
AUSTIN_LAT = 30.2672
AUSTIN_LON = -97.7431

# Kepler.gl uses a JSON-like dictionary for configuration.
# Here we set the initial view (mapState) to center on Austin.
map_config = {
    "version": "v1",
    "config": {
        "visState": {
            "filters": [
                {
                    "dataId": ["CapMetro_Buses"],
                    "id": "8oj7szg7q",
                    "name": ["timestamp"],
                    "type": "timeRange",
                    "value": [1772610478810.5244, 1772610594810.5244],
                    "plotType": {
                        "interval": "15-minute",
                        "defaultTimeFormat": "L  LT",
                        "type": "histogram",
                        "aggregation": "sum"
                    },
                    "animationWindow": "free",
                    "yAxis": None,
                    "view": "enlarged",
                    "speed": 0.22,
                    "enabled": True
                }
            ],
            "layers": [
                {
                    "id": "lk8579l",
                    "type": "point",
                    "config": {
                        "dataId": "CapMetro_Buses",
                        "columnMode": "points",
                        "label": "Point",
                        "color": [231, 159, 213],
                        "highlightColor": [252, 242, 26, 255],
                        "columns": { "lat": "latitude", "lng": "longitude" },
                        "isVisible": True,
                        "visConfig": {
                            "radius": 5.5,
                            "fixedRadius": False,
                            "opacity": 0.8,
                            "outline": False,
                            "thickness": 2,
                            "strokeColor": None,
                            "colorRange": {
                                "colors": [
                                    "#FF4040",
                                    "#F8671F",
                                    "#E49109",
                                    "#C5B900",
                                    "#9FDB05",
                                    "#75F317",
                                    "#4CFE34",
                                    "#29FC59",
                                    "#0FEC83",
                                    "#02D1AC",
                                    "#02ACD1",
                                    "#0F83EC",
                                    "#2959FC",
                                    "#4C34FE",
                                    "#7517F3",
                                    "#9F05DB",
                                    "#C500B9",
                                    "#E40991",
                                    "#F81F67",
                                    "#FF4040"
                                ],
                                "name": "Sinebow",
                                "type": "cyclical",
                                "category": "D3"
                            },
                            "strokeColorRange": {
                                "name": "Global Warming",
                                "type": "sequential",
                                "category": "Uber",
                                "colors": [
                                    "#4C0035",
                                    "#880030",
                                    "#B72F15",
                                    "#D6610A",
                                    "#EF9100",
                                    "#FFC300"
                                ]
                            },
                            "radiusRange": [0, 50],
                            "filled": True,
                            "billboard": False,
                            "allowHover": True,
                            "showNeighborOnHover": False,
                            "showHighlightColor": True
                        },
                        "hidden": False,
                        "textLabel": [
                            {
                                "field": None,
                                "color": [255, 255, 255],
                                "size": 18,
                                "offset": [0, 0],
                                "anchor": "start",
                                "alignment": "center",
                                "outlineWidth": 0,
                                "outlineColor": [255, 0, 0, 255],
                                "background": False,
                                "backgroundColor": [0, 0, 200, 255]
                            }
                        ]
                    },
                    "visualChannels": {
                        "colorField": { "name": "route_id", "type": "integer" },
                        "colorScale": "quantile",
                        "strokeColorField": None,
                        "strokeColorScale": "quantile",
                        "sizeField": None,
                        "sizeScale": "linear"
                    }
                }
            ],
            "effects": [],
            "interactionConfig": {
                "tooltip": {
                    "fieldsToShow": {
                        "CapMetro_Buses": [
                            { "name": "route_id", "format": None },
                            { "name": "trip_id", "format": None },
                            { "name": "vehicle_id", "format": None },
                            { "name": "speed", "format": None }
                        ]
                    },
                    "compareMode": False,
                    "compareType": "absolute",
                    "enabled": True
                },
                "brush": { "size": 0.5, "enabled": False },
                "geocoder": { "enabled": False },
                "coordinate": { "enabled": False }
            },
            "layerBlending": "normal",
            "overlayBlending": "normal",
            "splitMaps": [],
            "animationConfig": { "currentTime": None, "speed": 1 },
            "editor": { "features": [], "visible": True }
        },
        "mapState": {
            "bearing": 0,
            "dragRotate": False,
            "latitude": 30.27866105763493,
            "longitude": -97.7753201488513,
            "pitch": 0,
            "zoom": 10.16881985509215,
            "isSplit": False,
            "isViewportSynced": True,
            "isZoomLocked": False,
            "splitMapViewports": []
        },
        "mapStyle": {
            "styleType":"dark",
            "topLayerGroups": {},
            "visibleLayerGroups": {
                "label":False,
                "road":True,
                "border":False,
                "building":True,
                "water":True,
                "land":True,
                "3d building":False
            },
            "threeDBuildingColor":[9.665468314072013,17.18305478057247,31.1442867897876],
            "backgroundColor":[0,0,0],
            "mapStyles":{}
        },
        "uiState": { "mapControls": { "mapLegend": { "active": False } } }
    }
}


# Now initialize with this config
austin_map = KeplerGl(height=600, data={"CapMetro_Buses": df}, config=map_config)
print("Done!")

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter
Done!


In [7]:
file_name = 'capmetro_map_wconfig.html'
austin_map.save_to_html(file_name=file_name)

file_path = 'file://' + os.path.realpath(file_name)
webbrowser.open_new_tab(file_path)

Map saved to capmetro_map_wconfig.html!


True

In [22]:
import pydeck as pdk

In [26]:
trips_data = df.groupby('trip_id').apply(lambda x: {
    "waypoints": x[['longitude', 'latitude']].values.tolist(),
    "timestamps": x['raw_timestamp'].tolist(),
    "route": str(x['route_id'].iloc[0])
}, include_groups=False).tolist()

In [27]:
# 2. Define the Animated Layer
layer = pdk.Layer(
    "TripsLayer",
    trips_data,
    get_path="waypoints",
    get_timestamps="timestamps",
    get_color=[255, 180, 0], # Gold trails
    opacity=0.8,
    width_min_pixels=5,
    rounded=True,
    trail_length=600,  # 10 minutes of "tail"
    current_time=int(df['raw_timestamp'].min()), # Animation start point
)

# 3. View State (Austin Center)
view_state = pdk.ViewState(
    latitude=30.2672, 
    longitude=-97.7431, 
    zoom=12, 
    pitch=40
)

In [ ]:
r = pdk.Deck(
    layers=[layer], 
    initial_view_state=view_state,
    # Use Carto Dark Matter (No API key required)
    map_style='https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json'
)
r.to_html("pydeck_austin_buses.html", open_browser=True)